In [1]:
%run ../../Utils/yp_utils.py

# Initial setup

In [2]:
paper_pmid = 37572348
paper_name = 'cachera_mortensen_2023' 

In [3]:
datasets = pd.read_csv('extras/YeastPhenome_' + str(paper_pmid) + '_datasets_list.txt', sep='\t', header=None, names=['dataset_id', 'name'])

In [4]:
datasets.set_index('dataset_id', inplace=True)

# Load & process the data

In [5]:
original_data = pd.read_csv('raw_data/GA1_2_4_6.csv')

In [6]:
print('Original data dimensions: %d x %d' % (original_data.shape))

Original data dimensions: 4788 x 27


In [7]:
original_data.head()

,Unnamed: 0.1,Unnamed: 0,gene,area.24_mean,area.24_std,area.24_count,mean_intensity.24_mean,mean_intensity.24_std,mean_intensity.24_count,area.48_mean,...,corrected_area.24_count,corrected_mean_intensity.24_mean,corrected_mean_intensity.24_std,corrected_mean_intensity.24_count,corrected_area.48_mean,corrected_area.48_std,corrected_area.48_count,corrected_mean_intensity.48_mean,corrected_mean_intensity.48_std,corrected_mean_intensity.48_count
0,0,0,0,310.622222,133.379881,90.0,0.376688,0.100179,90.0,330.482759,...,87.0,-0.224677,1.204787,87.0,-0.013747,1.383006,84.0,-0.367429,1.503881,84.0
1,1,1,AAC1,264.750000,46.427722,16.0,0.402730,0.112292,16.0,279.000000,...,16.0,0.130642,1.053702,16.0,0.513305,0.442399,16.0,0.429670,0.388312,16.0
2,2,2,AAC3,263.333333,113.005900,3.0,0.369827,0.027531,3.0,281.000000,...,3.0,-0.213202,0.615854,3.0,-0.180554,1.005484,3.0,-0.386927,0.921191,3.0
3,3,3,AAD3,227.866667,40.746370,15.0,0.376890,0.124261,15.0,237.933333,...,15.0,-0.329509,1.119522,15.0,-0.064377,0.709291,15.0,-0.458212,1.108680,15.0
4,4,4,AAD4,218.533333,70.778191,15.0,0.412939,0.047924,15.0,231.200000,...,15.0,0.130813,0.463562,15.0,-0.225554,0.620656,15.0,0.401354,0.414679,15.0


In [8]:
original_data['gene'] = original_data['gene'].astype(str)

In [9]:
# Eliminate all white spaces & capitalize
original_data['gene'] = clean_genename(original_data['gene'])

In [10]:
# Translate to ORFs 
original_data['orf'] = translate_sc(original_data['gene'], to='orf')

In [11]:
original_data.loc[original_data['orf']=='YLR287-A','orf'] = 'YLR287C-A'

In [12]:
# Make sure everything translated ok
t = looks_like_orf(original_data['orf'])
print(original_data.loc[~t,])

             Unnamed: 0.1  Unnamed: 0  gene  area.24_mean  area.24_std  \
index_input                                                              
0                       0           0     0    310.622222   133.379881   
5                       5           5  AAD6    253.500000    55.904681   
589                   589         589  CRS5    326.000000   115.215740   
945                   945         945  FLO8    254.428571    65.578742   
3479                 3479        3479    WT    269.000000    79.653796   

             area.24_count  mean_intensity.24_mean  mean_intensity.24_std  \
index_input                                                                 
0                     90.0                0.376688               0.100179   
5                     16.0                0.415353               0.046135   
589                   16.0                0.414568               0.120935   
945                   14.0                0.418740               0.131385   
3479               

In [13]:
original_data = original_data.loc[t,]

In [14]:
original_data.set_index('orf', inplace=True)

In [15]:
data_fields = ['corrected_mean_intensity.24_mean','corrected_mean_intensity.48_mean']
original_data = original_data[data_fields].copy()

In [16]:
original_data = original_data.groupby(original_data.index).mean()

In [17]:
original_data.shape

(4756, 2)

# Prepare the final dataset

In [18]:
data = original_data.copy()

In [19]:
dataset_ids = [22286, 22288]
datasets = datasets.reindex(index=dataset_ids)

In [20]:
lst = [datasets.index.values, ['value']*datasets.shape[0]]
tuples = list(zip(*lst))
idx = pd.MultiIndex.from_tuples(tuples, names=['dataset_id','data_type'])
data.columns = idx

In [21]:
data.head()

dataset_id,22286,22288
data_type,value,value
orf,,
YAL002W,-0.567050,-0.762672
YAL004W,-0.138001,-0.312376
YAL005C,-0.091220,0.171002
YAL007C,0.208304,0.031030
YAL008W,-0.648933,-0.540021


## Subset to the genes currently in SGD

In [22]:
genes = pd.read_csv(path_to_genes, sep='\t', index_col='id')
genes = genes.reset_index().set_index('systematic_name')
gene_ids = genes.reindex(index=data.index.values)['id'].values
num_missing = np.sum(np.isnan(gene_ids))
print('ORFs missing from SGD: %d' % num_missing)

ORFs missing from SGD: 20


In [23]:
data['gene_id'] = gene_ids
data = data.loc[data['gene_id'].notnull()]
data['gene_id'] = data['gene_id'].astype(int)
data = data.reset_index().set_index(['gene_id','orf'])

data.head()

,dataset_id,22286,22288
,data_type,value,value
gene_id,orf,,
2,YAL002W,-0.567050,-0.762672
1863,YAL004W,-0.138001,-0.312376
4,YAL005C,-0.091220,0.171002
5,YAL007C,0.208304,0.031030
6,YAL008W,-0.648933,-0.540021


# Normalize

In [24]:
data_norm = normalize_phenotypic_scores(data, has_tested=True)

In [25]:
# Assign proper column names
lst = [datasets.index.values, ['valuez']*datasets.shape[0]]
tuples = list(zip(*lst))
idx = pd.MultiIndex.from_tuples(tuples, names=['dataset_id','data_type'])
data_norm.columns = idx

In [26]:
data_norm[data.isnull()] = np.nan
data_all = data.join(data_norm)

data_all.head()

,dataset_id,22286,22288,22286,22288
,data_type,value,value,valuez,valuez
gene_id,orf,,,,
2,YAL002W,-0.567050,-0.762672,-1.010730,-1.254715
1863,YAL004W,-0.138001,-0.312376,-0.278456,-0.606504
4,YAL005C,-0.091220,0.171002,-0.198612,0.089328
5,YAL007C,0.208304,0.031030,0.312598,-0.112163
6,YAL008W,-0.648933,-0.540021,-1.150484,-0.934205


# Print out

In [27]:
for f in ['value','valuez']:
    df = data_all.xs(f, level='data_type', axis=1).copy()
    df.columns = datasets['name'].values
    df = df.droplevel('gene_id', axis=0)
    df.to_csv(paper_name + '_' + f + '.txt', sep='\t')